# TD 9 — Peut-on prévoir un cours de bourse ?

**Analyse des données — L3 Économie**

Jusqu'ici vous avez cherché à **expliquer**. Aujourd'hui vous allez **prévoir**, et ce n'est pas le même exercice : on ne juge plus un modèle sur son $R^2$, mais sur ce qu'il fait face à des données **qu'il n'a jamais vues**.

> *Exécution → Tout exécuter* avant de démarrer.


## Partie 1 — Le piège : prévoir le prix

In [ ]:
!pip install -q yfinance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import yfinance as yf

TITRE  = "RNO.PA"      # Renault
MARCHE = "^FCHI"       # CAC 40

brut = yf.download([TITRE, MARCHE], start="2019-01-01", end="2025-12-31",
                   auto_adjust=True, progress=False)

px  = brut["Close"].dropna()
vlm = brut["Volume"].dropna()
px.columns  = ["MARCHE" if c == MARCHE else "TITRE" for c in px.columns]
vlm.columns = ["MARCHE" if c == MARCHE else "TITRE" for c in vlm.columns]

print(px.shape, "| du", px.index.min().date(), "au", px.index.max().date())
px.tail(3).round(2)

In [ ]:
# Secours si le telechargement echoue le jour du TD :
# px  = pd.read_csv("https://raw.githubusercontent.com/StefaniaMarcassa/"
#                   "analyse_des_donnees/main/data/td09-cours.csv",
#                   index_col="Date", parse_dates=True)

In [ ]:
r = np.log(px).diff().dropna()
print(len(r), "seances")

**Question 2.** Régressez le prix de demain sur celui d'aujourd'hui.

In [ ]:
p = pd.DataFrame({"demain": px["TITRE"].shift(-1), "aujourdhui": px["TITRE"]}).dropna()
mod = smf.ols("demain ~ aujourdhui", data=p).fit()
print("R2 =", round(mod.rsquared, 5))

*Si vous en jugiez le modèle, que concluriez-vous ?*

**Question 3.** Comparez-le maintenant à une règle sans aucun paramètre.


In [ ]:
# a vous : l'erreur quadratique moyenne du modele, puis celle de "demain = aujourd'hui"

*Laquelle gagne ? Que valait donc le $R^2$ ?*

## Partie 2 — Prévoir le rendement

**Question 4.** Les variables explicatives, toutes connues **la veille**.


In [ ]:
X = pd.DataFrame(index=px.index)

for k in range(1, 6):
    X[f"r_lag{k}"] = r["TITRE"].shift(k)          # les cinq derniers rendements

X["ma5"]     = r["TITRE"].shift(1).rolling(5).mean()
X["ma21"]    = r["TITRE"].shift(1).rolling(21).mean()
X["vol5"]    = r["TITRE"].shift(1).rolling(5).std()
X["vol21"]   = r["TITRE"].shift(1).rolling(21).std()
X["marche1"] = r["MARCHE"].shift(1)
X["lvolume"] = np.log(vlm["TITRE"]).shift(1)
X["lundi"]   = (X.index.dayofweek == 0).astype(int)

VARS = list(X.columns)
print(len(VARS), "variables :", VARS)

**Question 5.** Le découpage — dans le temps, jamais au hasard.

In [ ]:
def jeu(cible):
    """Assemble cible + variables, puis coupe 80 / 20 DANS LE TEMPS."""
    d = pd.concat([cible.rename("y"), X], axis=1).dropna()
    c = int(len(d) * 0.8)
    return d.iloc[:c], d.iloc[c:]

tr, te = jeu(r["TITRE"])
print("apprentissage :", tr.index[0].date(), "->", tr.index[-1].date(), f"  ({len(tr)})")
print("test          :", te.index[0].date(), "->", te.index[-1].date(), f"  ({len(te)})")

*Pourquoi pas au hasard ?*

**Questions 6 et 7.** Le modèle, puis les deux repères.


In [ ]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def r2_hors(y_vrai, y_prev, repere):
    """1 - SCR du modele / SCR du repere. Peut etre negatif."""
    y_vrai = np.asarray(y_vrai)
    return 1 - np.sum((y_vrai - y_prev)**2) / np.sum((y_vrai - repere)**2)

In [ ]:
m = LinearRegression().fit(tr[VARS], tr["y"])
prev = m.predict(te[VARS])

print("R2 en APPRENTISSAGE :", round(m.score(tr[VARS], tr["y"]), 4))
print()
print("RMSE du modele          :", round(rmse(te["y"], prev), 6))
print("RMSE du repere 'zero'   :", round(rmse(te["y"], np.zeros(len(te))), 6))
print("RMSE du repere 'moyenne':", round(rmse(te["y"], np.full(len(te), tr["y"].mean())), 6))
print()
print("R2 HORS ECHANTILLON (vs zero)    :", round(r2_hors(te["y"], prev, 0), 4))
print("R2 HORS ECHANTILLON (vs moyenne) :",
      round(r2_hors(te["y"], prev, tr["y"].mean()), 4))

*Commentez l'écart entre les deux $R^2$. Le $R^2$ hors échantillon peut-il être négatif, et qu'est-ce que cela voudrait dire ?*

## Partie 3 — Prévoir seulement le sens

**Questions 8 et 9.**


In [ ]:
hausse = (te["y"] > 0).mean()
juste  = ((prev > 0) == (te["y"] > 0)).mean()
n      = len(te)
marge  = 1.96 * 0.5 / np.sqrt(n)          # incertitude sur une proportion

print("seances en hausse dans le test :", round(100 * hausse, 1), "%")
print("  -> 'toujours en hausse' a raison", round(100 * hausse, 1), "% du temps")
print("le modele a raison             :", round(100 * juste, 1), "%")
print()
print(f"incertitude sur {n} journees : +/- {100*marge:.1f} points")

*Le modèle fait-il mieux que le taux de base ? L'écart dépasse-t-il l'incertitude ?*

## Partie 4 — Ce qui, lui, est prévisible

**Question 10.** Le même code, une autre cible.


In [ ]:
tra, tea = jeu(r["TITRE"].abs())        # meme code, cible differente

ma = LinearRegression().fit(tra[VARS], tra["y"])
pa = ma.predict(tea[VARS])

print("cible = |rendement| (l'amplitude, sans le sens)")
print("  R2 en apprentissage      :", round(ma.score(tra[VARS], tra["y"]), 4))
print("  R2 HORS ECHANTILLON      :", round(r2_hors(tea["y"], pa, tra["y"].mean()), 4))

**Question 11.** Pourquoi l'amplitude est-elle prévisible alors que le sens ne l'est pas ? Et pourquoi n'est-ce pas contradictoire avec l'efficience des marchés ?

**Question 12.** Trois phrases : ce que vous avez prévu, ce que vous n'avez pas prévu, et pourquoi le second résultat est le plus intéressant.

*Vos réponses :*

## Pour ceux qui ont terminé

**Question 13.** Retirez le `.shift(1)` de `vol5` et refaites la partie 4.

**Question 14.** Vingt variables tirées au hasard : combien sont « significatives » ?

**Question 15.** Remplacez `LinearRegression` par `RandomForestRegressor`, sans rien changer d'autre.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
# a vous